# 索引与切片 (Indexing & Slicing)

与 NumPy 的语法几乎完全一致。

- 基本切片：tensor[0] (第一行), tensor[:, 1] (所有行的第 1 列), tensor[...,-1] (最后一列)。
- 布尔索引：tensor[tensor > 0] (选取大于 0 的元素)。
- 高级索引：
    - torch.index_select(input, dim, index): 沿指定维度选取索引。
    - torch.masked_select(input, mask): 根据掩码选取元素。
    - torch.gather(dim, input, index): 沿指定维度收集值。

## 🔪 基本切片 (Basic Slicing)

- 基本切片使用 Python 的切片语法，通过冒号 :、逗号 , 和省略号 ... 来访问张量的特定部分。
- 特点：操作非常高效，因为<font color='red'>它返回的是原始张量的一个视图（View），与原始数据共享内存。修改切片后的张量会直接影响原始张量。</font>

In [ ]:
import torch
x = torch.arange(12).reshape(3, 4)
# x = tensor([[ 0,  1,  2,  3],
#             [ 4,  5,  6,  7],
#             [ 8,  9, 10, 11]])


# 选取第一行
row_0 = x[0]  # tensor([0, 1, 2, 3])

# 选取所有行的第一列
col_1 = x[:,1] # tensor([1,5,9])

# 选取最后一列，... 代表前面所有维度
last_col = x[...,-1]  # tensor([ 3,  7, 11])

## 🎭 布尔索引 (Boolean Indexing)
- 布尔索引<font color='red'>使用一个布尔张量（掩码）作为索引，来选取满足特定条件的元素。这个布尔张量通常由比较运算（如 >, <, ==）生成。</font>
- 特点：非常灵活，可以根据任意条件筛选数据。<font color='red'>它返回的是一个副本（Copy），而不是视图，因为选出的元素在原始张量中可能不是连续的。</font>

In [2]:
import torch
x = torch.tensor([[1, -2, 3], [-4, 5, -6]])

# 创建一个布尔掩码
mask = x > 0

# mask = tensor([[ True, False,  True],
#                [False,  True, False]])

# 使用布尔掩码选取所有大于0的元素
positive_elements = x[mask] # tensor([1,3,5])

positive_elements_1 = x[x>0]
print(positive_elements_1)

tensor([1, 3, 5])


## 🚀 高级索引 (Advanced Indexing)

高级索引提供了比基本切片更强大的数据选取能力，通常通过专门的函数实现。与基本切片不同，高级索引返回的通常是数据的副本（Copy）。

### torch.index_select(input, dim, index)
- <font color='red'>沿指定维度 dim，根据索引张量 index 选取数据。</font>
- 功能：从 input 张量的 dim 维度上，选取 index 中指定的索引对应的数据。index 必须是一维的。

In [3]:
x = torch.arange(12).reshape(3,4)
#tensor([[ 0,  1,  2,  3],
#        [ 4,  5,  6,  7],
#        [ 8,  9, 10, 11]])

print(x)

# 沿第0维（行）选取索引为0和2的两行
indices = torch.tensor([0,2])
y =torch.index_select(x,0,indices)
print(y)
 
# tensor([[ 0,  1,  2,  3],
#        [ 8,  9, 10, 11]])   

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])


### torch.masked_select(input, mask)
- <font color='red'>根据布尔掩码 mask 选取 input 中所有对应为 True 的元素。</font>
- 功能：功能上与布尔索引 input[mask] 等价，但以函数形式提供。返回一个包含所有选中元素的一维张量。

In [4]:
x = torch.tensor([[1, -2, 3], [-4, 5, -6]])

mask = x > 2
y = torch.masked_select(x,mask)
print(y)

tensor([3, 5])


### <font color='red'>torch.gather(dim, input, index)</font>
- <font color='red'>沿指定维度 dim，根据 index 张量中的索引，从 input 中收集值。这是最灵活也最复杂的一个。</font>
- 功能：index 和 input 必须有相同的维度数。它根据 index 中的值，在 input 的指定 dim 维度上查找并收集元素。输出张量的形状与 index 相同。

In [5]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])

# index指定了要收集的列索引
index = torch.tensor([[0, 2], [1, 0]]) # # 第0行取第0、2列；第1行取第1、0列
# 沿第 1 维（列）收集

y = torch.gather(x,1,index)
print(y)

tensor([[1, 3],
        [5, 4]])


| 方法 | 核心功能 | 内存共享情况 | 适用场景 |
| :--- | :--- | :--- | :--- |
| 基本切片 | 访问连续子区域 | 总是共享 (视图) | 提取行、列或数据块 |
| 布尔索引 | 按条件筛选元素 | 总是复制 (副本) | 数据过滤、条件筛选 |
| 高级索引 | 按指定索引选取 | 通常复制 (副本) | 复杂的、非连续的元素选择 |
[(video_note_list_1)]